# 🩻 Demo Tương Tác PGA-UNet (Prompt-Guided Attention U-Net)

Notebook này dựng lại một quy trình demo tương tác cho mô hình **PGA-UNet**, dùng để chạy trên Google Colab / Kaggle:

1. **Setup**: tải kiến trúc PGA-UNet từ GitHub, tải 2 bộ dữ liệu test (BTXRD, FracAtlas) và 2 bộ trọng số tương ứng từ Google Drive.
2. **Chọn ảnh**: chọn 1 ảnh trong tập test (đã có sẵn Ground Truth) — vì cần GT để tính 6 độ đo, notebook chọn ảnh từ tập test có sẵn thay vì cho tải ảnh bất kỳ không có nhãn. Vì có **2 bộ trọng số** (mỗi bộ huấn luyện riêng cho 1 dataset), giao diện có 1 ô chọn ảnh này thuộc **BTXRD hay FracAtlas** (mặc định khớp bộ trọng số theo đúng dataset, nhưng có thể đổi độc lập để thử tổng quát hóa liên miền dữ liệu, ví dụ dùng trọng số BTXRD chạy trên ảnh FracAtlas).
3. **Khoanh vùng**: click 2 điểm trên ảnh để tạo Box, nhấn **Xác Nhận** → Box được chuyển thành *Prompt Heatmap* (giống hệt tiền xử lý trong `dataset.py`) → đưa vào PGA-UNet → nhận mask dự đoán.
4. **Đối chiếu**: so mask dự đoán với Ground Truth, hiển thị đủ **6 độ đo**: Dice, IoU, Precision, Recall, HD95, CBL.
5. **Tiếp tục / Kết thúc**:
   - **Tiếp tục**: dùng lại đúng ảnh đó, khoanh thêm 1 Box (prompt) mới → mask mới được **hợp (union)** với (các) mask đã lưu trước đó → tính lại 6 độ đo trên ảnh + mask hợp nhất. Có thể lặp lại nhiều vòng.
   - **Kết thúc**: xóa sạch ảnh/điểm/mask đang xử lý, quay lại bước chọn ảnh để bắt đầu với ảnh mới.

> **Lưu ý phạm vi**: để tính được 6 độ đo cần Ground Truth, nên demo chỉ thao tác trên các ảnh **test** đã có annotation JSON sẵn trong 2 bộ dữ liệu, không phải upload ảnh ngoài chưa có nhãn.


**Cell 1 — Setup môi trường + clone kiến trúc PGA-UNet**

In [5]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP MÔI TRƯỜNG (Colab / Kaggle)
# ══════════════════════════════════════════════════════
import os

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

REPO_PGA = 'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'

if not os.path.exists(f'{BASE}/pga-repo'):
    !git clone -q -b TN_B_ON {REPO_PGA} {BASE}/pga-repo
print('✅ Đã clone kiến trúc PGA-UNet (branch TN_B_ON)')

!pip install -q gdown opencv-python matplotlib scikit-image gradio tqdm scipy

os.chdir(f'{BASE}/pga-repo')
print('✅ Setup xong, thư mục làm việc:', os.getcwd())

# ── Hàm tải Google Drive có fallback, tránh lỗi FileURLRetrievalError ──
# (gdown ẩn danh hay bị Google chặn nếu file bị tải nhiều lần trong ngày hoặc
#  chưa share đúng "Anyone with the link" — fallback dùng Drive API qua chính
#  tài khoản Google đăng nhập Colab để tải, không phụ thuộc giới hạn ẩn danh)
def robust_gdrive_download(file_id, output_path, quiet=False):
    import gdown
    if os.path.exists(output_path):
        return
    url = f'https://drive.google.com/uc?id={file_id}'
    try:
        gdown.download(url, output_path, quiet=quiet)
        if os.path.exists(output_path):
            return
    except Exception as e:
        print(f'  ⚠️ gdown thất bại ({e}); thử lại với fuzzy=True...')
    try:
        gdown.download(id=file_id, output=output_path, quiet=quiet, fuzzy=True)
        if os.path.exists(output_path):
            return
    except Exception as e:
        print(f'  ⚠️ gdown fuzzy cũng thất bại ({e}); thử fallback Drive API (tài khoản Colab)...')
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        auth.authenticate_user()
        service = build('drive', 'v3')
        request = service.files().get_media(fileId=file_id)
        with open(output_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        print(f'  ✅ Tải thành công qua Drive API (tài khoản Colab): {output_path}')
        return
    except Exception as e:
        raise RuntimeError(
            f"❌ Không tải được file Google Drive (id={file_id}).\n"
            f"Nguyên nhân thường gặp:\n"
            f"  1. File chưa được share 'Anyone with the link' (Viewer).\n"
            f"  2. File đã vượt giới hạn tải công khai trong ngày của Google Drive — thử lại sau vài giờ,\n"
            f"     hoặc tải thủ công qua trình duyệt tại https://drive.google.com/uc?id={file_id}\n"
            f"     rồi upload trực tiếp vào {output_path}.\n"
            f"  3. Không chạy trên Google Colab nên không dùng được xác thực tài khoản (fallback 3).\n"
            f"Lỗi Drive API gốc: {e}"
        ) from e


✅ Đã clone kiến trúc PGA-UNet (branch TN_B_ON)
✅ Setup xong, thư mục làm việc: /kaggle/working/pga-repo


**Cell 2 — Tải 2 bộ dữ liệu test (BTXRD + FracAtlas) từ Google Drive**

In [6]:
# ══════════════════════════════════════════════════════
# CELL 2 — TẢI 2 BỘ DỮ LIỆU TEST (BTXRD + FracAtlas) TỪ GOOGLE DRIVE
# ══════════════════════════════════════════════════════
import os

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'

# File ID Google Drive của 2 bộ dữ liệu (zip chứa sẵn split train/val/test)
DATASETS_GDRIVE = {
    'BTXRD':     '1y7wD82n51hdLcbwrQwFtm-7sBnD_OJet',
    'FracAtlas': '1o4WUbs9faT6T-F-i59tPKSQNaROBY4DL',
}

for name, file_id in DATASETS_GDRIVE.items():
    ds_path = f'{BASE}/pga-repo/dataset_{name}'
    if not os.path.exists(ds_path):
        zip_path = f'{BASE}/dataset_{name}.zip'
        robust_gdrive_download(file_id, zip_path, quiet=False)
        !unzip -oq {zip_path} -d {BASE}/pga-repo/
    print(f'✅ {name}: {ds_path}')


✅ BTXRD: /kaggle/working/pga-repo/dataset_BTXRD
✅ FracAtlas: /kaggle/working/pga-repo/dataset_FracAtlas


**Cell 3 — Tải 2 bộ trọng số PGA-UNet (512×512), mỗi bộ ứng với 1 dataset**

In [7]:
# ══════════════════════════════════════════════════════
# CELL 3 — TẢI TRỌNG SỐ PGA-UNET (512×512) TỪ GOOGLE DRIVE
# ══════════════════════════════════════════════════════
import os

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
CKPT_DIR = f'{BASE}/pga-repo/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Mỗi dataset có 1 checkpoint PGA-UNet-512 huấn luyện riêng trên chính dataset đó
CHECKPOINTS_GDRIVE = {
    'BTXRD':     ('13_51tUHcFSu85GqJTri0hDrPGcInHYBz', 'pga_btxrd_512_best.pth'),
    'FracAtlas': ('1xS_3ukhTF7CrcX_4AqcHcIDFFqQn2ptR', 'pga_fracatlas_512_best.pth'),
}

CKPT_PATHS = {}
for name, (file_id, fname) in CHECKPOINTS_GDRIVE.items():
    fpath = os.path.join(CKPT_DIR, fname)
    robust_gdrive_download(file_id, fpath, quiet=False)
    CKPT_PATHS[name] = fpath
    print(f'✅ {name}: {fpath}  ({os.path.getsize(fpath)//1024} KB)')


Downloading...
From: https://drive.google.com/uc?id=13_51tUHcFSu85GqJTri0hDrPGcInHYBz
To: /kaggle/working/pga-repo/checkpoints/pga_btxrd_512_best.pth
100%|██████████| 11.9M/11.9M [00:00<00:00, 37.9MB/s]


✅ BTXRD: /kaggle/working/pga-repo/checkpoints/pga_btxrd_512_best.pth  (11615 KB)


Downloading...
From: https://drive.google.com/uc?id=1xS_3ukhTF7CrcX_4AqcHcIDFFqQn2ptR
To: /kaggle/working/pga-repo/checkpoints/pga_fracatlas_512_best.pth
100%|██████████| 11.9M/11.9M [00:00<00:00, 57.6MB/s]

✅ FracAtlas: /kaggle/working/pga-repo/checkpoints/pga_fracatlas_512_best.pth  (11615 KB)


**Cell 4 — Nạp kiến trúc PGA-UNet + các hàm xử lý dùng chung (tiền xử lý, heatmap, 6 độ đo)**

In [8]:
# ══════════════════════════════════════════════════════
# CELL 4 — NẠP MÔ HÌNH + HÀM XỬ LÝ DÙNG CHUNG
# ══════════════════════════════════════════════════════
import sys, json, cv2
import numpy as np
import torch
from scipy.ndimage import binary_erosion, distance_transform_edt

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
if f'{BASE}/pga-repo' not in sys.path:
    sys.path.insert(0, f'{BASE}/pga-repo')

from models.networks.prompt_unet_2D import PGA_UNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CANVAS = 512  # khớp đúng kích thước 2 checkpoint đã tải ở Cell 3

DATASET_DIRS = {
    'BTXRD':     dict(img_dir=f'{BASE}/pga-repo/dataset_BTXRD/test/images',
                       json_dir=f'{BASE}/pga-repo/dataset_BTXRD/test/annotations'),
    'FracAtlas': dict(img_dir=f'{BASE}/pga-repo/dataset_FracAtlas/test/images',
                       json_dir=f'{BASE}/pga-repo/dataset_FracAtlas/test/annotations'),
}

MODELS = {}
for name, ckpt_path in CKPT_PATHS.items():
    m = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    m.eval()
    MODELS[name] = m
    print(f'✅ PGA-UNet ({name}) sẵn sàng — device={DEVICE}')

# ── Tiền xử lý: PHẢI khớp chính xác Prompt-Guided-XRay-Segmentation/dataset.py ──
# (resize giữ tỉ lệ khung hình rồi pad vuông, và plateau-heatmap Gaussian k=31)
def resize_and_pad(array, size, interpolation, pad_value=0):
    orig_h, orig_w = array.shape[:2]
    scale = min(size / orig_w, size / orig_h)
    new_w, new_h = max(1, int(round(orig_w * scale))), max(1, int(round(orig_h * scale)))
    resized = cv2.resize(array, (new_w, new_h), interpolation=interpolation)
    padded = np.full((size, size), pad_value, dtype=resized.dtype)
    pad_left, pad_top = (size - new_w) // 2, (size - new_h) // 2
    padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
    return padded

def create_plateau_heatmap(bbox, size):
    heatmap = np.zeros((size, size), dtype=np.float32)
    x_min, y_min, x_max, y_max = bbox
    x_min, y_min = max(0, int(x_min - 5)), max(0, int(y_min - 5))
    x_max, y_max = min(size, int(x_max + 5)), min(size, int(y_max + 5))
    if x_max > x_min and y_max > y_min:
        heatmap[y_min:y_max, x_min:x_max] = 1.0
        heatmap = cv2.GaussianBlur(heatmap, (31, 31), 0)
    return heatmap

# ── 6 độ đo — khớp chính xác cách tính trong Result/*/test-pga-*.ipynb ──
def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(CANVAS)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(CANVAS) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_metrics_img(prob_np, gt_np, eps=1e-6):
    pm = (prob_np > 0.5).astype(np.float32)
    gm = (gt_np   > 0.5).astype(np.float32)
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    hd95 = calc_hd95(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        bbox_diag = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2) / bbox_diag, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)), recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

METRIC_HDRS = ['Dice ↑', 'IoU ↑', 'Precision ↑', 'Recall ↑', 'HD95 ↓ (px)', 'CBL ↑']

# ── Danh sách ảnh test có Ground Truth + nạp canvas 512×512 ──
def list_test_images(dataset_key):
    dirs = DATASET_DIRS[dataset_key]
    names = []
    for fn in sorted(os.listdir(dirs['img_dir'])):
        if not fn.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        base = os.path.splitext(fn)[0]
        if os.path.exists(os.path.join(dirs['json_dir'], base + '.json')):
            names.append(fn)
    return names

def load_canvas(dataset_key, img_name):
    # Trả về (canvas_img uint8, canvas_gt float32 0/1) ở đúng không gian 512x512 mô hình nhìn thấy.
    dirs = DATASET_DIRS[dataset_key]
    base = os.path.splitext(img_name)[0]
    img = cv2.imread(os.path.join(dirs['img_dir'], img_name), cv2.IMREAD_GRAYSCALE)
    orig_h, orig_w = img.shape

    gt_native = np.zeros((orig_h, orig_w), dtype=np.uint8)
    with open(os.path.join(dirs['json_dir'], base + '.json'), 'r', encoding='utf-8') as f:
        data = json.load(f)
    for s in data.get('shapes', []):
        if s.get('shape_type') == 'polygon':
            cv2.fillPoly(gt_native, [np.array(s['points'], dtype=np.int32)], 1)

    canvas_img = resize_and_pad(img, CANVAS, cv2.INTER_LINEAR, pad_value=0)
    canvas_gt  = resize_and_pad(gt_native.astype(np.float32), CANVAS, cv2.INTER_NEAREST, pad_value=0.0)
    canvas_gt  = (canvas_gt > 0.5).astype(np.float32)
    return canvas_img, canvas_gt

def run_prompt(dataset_key, canvas_img, bbox):
    # bbox theo tọa độ canvas 512x512 → heatmap → PGA-UNet → mask nhị phân 512x512.
    heatmap   = create_plateau_heatmap(bbox, CANVAS)
    img_t     = torch.from_numpy((canvas_img.astype(np.float32) / 255.0 - 0.5) / 0.5).unsqueeze(0).unsqueeze(0).to(DEVICE)
    heatmap_t = torch.from_numpy(heatmap).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(MODELS[dataset_key](img_t, heatmap_t))
    return (prob[0, 0].cpu().numpy() > 0.5).astype(np.float32)

def build_overlay(canvas_img_gray, acc_mask, canvas_gt, boxes, show_gt):
    base = cv2.cvtColor(canvas_img_gray, cv2.COLOR_GRAY2RGB).astype(np.float32)
    result = base.copy()
    if show_gt and canvas_gt is not None and canvas_gt.max() > 0:
        green = np.zeros_like(result); green[..., 1] = canvas_gt * 255
        result = cv2.addWeighted(result, 1.0, green, 0.35, 0)
    if acc_mask is not None and acc_mask.max() > 0:
        red = np.zeros_like(result); red[..., 0] = acc_mask * 255
        result = cv2.addWeighted(result, 1.0, red, 0.45, 0)
    result = np.clip(result, 0, 255).astype(np.uint8)
    for (bx1, by1, bx2, by2) in boxes:
        cv2.rectangle(result, (int(bx1), int(by1)), (int(bx2), int(by2)), (255, 255, 0), 1)
    return result

def metrics_to_markdown(m, round_idx, n_prompts, weight_key):
    return (f"### 📊 Kết quả sau {n_prompts} câu nhắc (vòng {round_idx}) — Trọng số PGA-UNet: **{weight_key}**\n\n"
            f"| {' | '.join(METRIC_HDRS)} |\n"
            f"|{'---|' * len(METRIC_HDRS)}\n"
            f"| {m['dice']:.4f} | {m['iou']:.4f} | {m['precision']:.4f} | "
            f"{m['recall']:.4f} | {m['hd95']:.2f} | {m['cbl']:.4f} |")

print('✅ Helper functions sẵn sàng')


✅ PGA-UNet (BTXRD) sẵn sàng — device=cuda
✅ PGA-UNet (FracAtlas) sẵn sàng — device=cuda
✅ Helper functions sẵn sàng


**Cell 5 — Giao diện Gradio tương tác (click 2 điểm → Box → mask → 6 độ đo → Tiếp tục / Kết thúc)**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5 — GIAO DIỆN GRADIO TƯƠNG TÁC
# ══════════════════════════════════════════════════════
import warnings
warnings.filterwarnings("ignore")
import gradio as gr

EMPTY_CANVAS_MSG = "⬅️ Chọn bộ dữ liệu + ảnh test rồi nhấn 'Tải ảnh' để bắt đầu."


def on_dataset_change(dataset_key):
    names = list_test_images(dataset_key)
    img_update = gr.update(choices=names, value=names[0] if names else None)
    weight_update = gr.update(value=dataset_key)  # mặc định khớp trọng số theo dataset, có thể đổi tay sau đó
    return img_update, weight_update


def on_load_image(dataset_key, img_name):
    if not img_name:
        return (None, None, None, [], np.zeros((CANVAS, CANVAS), dtype=np.float32), [], 0,
                "⚠️ Không tìm thấy ảnh nào có Ground Truth.", None, "",
                gr.update(visible=False), gr.update(visible=False))
    canvas_img, canvas_gt = load_canvas(dataset_key, img_name)
    zero_mask = np.zeros((CANVAS, CANVAS), dtype=np.float32)
    display = build_overlay(canvas_img, zero_mask, canvas_gt, [], show_gt=False)
    status = "🎯 Click điểm 1, rồi click điểm 2 để tạo Box, sau đó nhấn 'Xác Nhận'."
    return (display, canvas_img, canvas_gt, [], zero_mask, [], 0,
            status, None, "",
            gr.update(visible=False), gr.update(visible=False))


def _redraw_with_points(canvas_img, acc_mask, canvas_gt, boxes, points, show_gt):
    display = build_overlay(canvas_img, acc_mask, canvas_gt, boxes, show_gt)
    for p in points:
        cv2.circle(display, p, 6, (255, 60, 60), -1)
        cv2.circle(display, p, 7, (255, 255, 255), 1)
    if len(points) == 2:
        (px1, py1), (px2, py2) = points
        cv2.rectangle(display, (min(px1, px2), min(py1, py2)), (max(px1, px2), max(py1, py2)), (50, 220, 50), 2)
    return display


def on_click(evt: gr.SelectData, canvas_img, acc_mask, canvas_gt, boxes, points, show_gt):
    if canvas_img is None:
        return gr.update(), points, "⚠️ Vui lòng tải ảnh trước."
    pts = list(points)
    if len(pts) >= 2:
        pts = []
    pts.append((int(evt.index[0]), int(evt.index[1])))
    display = _redraw_with_points(canvas_img, acc_mask, canvas_gt, boxes, pts, show_gt)
    if len(pts) == 2:
        status = "🎯 Đã khoanh xong Box. Nhấn 'Xác Nhận' để phân đoạn."
    else:
        status = f"Đã lấy góc 1: {pts[-1]}. Click góc 2 để hoàn tất Box."
    return display, pts, status


def on_reset_points(canvas_img, acc_mask, canvas_gt, boxes, show_gt):
    if canvas_img is None:
        return gr.update(), [], "⚠️ Vui lòng tải ảnh trước."
    display = build_overlay(canvas_img, acc_mask, canvas_gt, boxes, show_gt)
    return display, [], "🔄 Đã đặt lại điểm. Click 2 điểm mới để khoanh Box."


def on_toggle_gt(canvas_img, acc_mask, canvas_gt, boxes, points, show_gt):
    if canvas_img is None:
        return gr.update()
    return _redraw_with_points(canvas_img, acc_mask, canvas_gt, boxes, points, show_gt)


def on_confirm(points, canvas_img, canvas_gt, acc_mask, boxes, weight_key, round_idx, show_gt):
    if canvas_img is None:
        return (gr.update(), gr.update(), acc_mask, boxes, round_idx,
                gr.update(), points, "⚠️ Vui lòng tải ảnh trước.",
                gr.update(visible=False), gr.update(visible=False))
    if len(points) < 2:
        return (gr.update(), gr.update(), acc_mask, boxes, round_idx,
                gr.update(), points, "⚠️ Cần click đủ 2 điểm để tạo Box.",
                gr.update(visible=False), gr.update(visible=False))

    (x1, y1), (x2, y2) = points
    bbox = [min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)]
    new_mask = run_prompt(weight_key, canvas_img, bbox)

    acc_mask_new = np.maximum(acc_mask, new_mask)
    boxes_new = boxes + [tuple(bbox)]
    round_new = round_idx + 1

    m = calc_metrics_img(acc_mask_new, canvas_gt)
    result_display = build_overlay(canvas_img, acc_mask_new, canvas_gt, boxes_new, show_gt)
    md_text = metrics_to_markdown(m, round_new, len(boxes_new), weight_key)
    base_display = build_overlay(canvas_img, acc_mask_new, canvas_gt, boxes_new, show_gt)
    status = (f"✅ Đã phân đoạn vòng {round_new}. Chọn 'Tiếp tục' để thêm câu nhắc, "
              f"hoặc 'Kết thúc' để đổi ảnh khác.")

    return (result_display, md_text, acc_mask_new, boxes_new, round_new,
            base_display, [], status, gr.update(visible=True), gr.update(visible=True))


def on_continue(canvas_img, acc_mask, canvas_gt, boxes, show_gt):
    display = build_overlay(canvas_img, acc_mask, canvas_gt, boxes, show_gt)
    status = "🎯 Click 2 điểm để khoanh Box bổ sung (câu nhắc mới), rồi nhấn 'Xác Nhận'."
    return display, [], status, gr.update(visible=False), gr.update(visible=False)


def on_end(dataset_key):
    names = list_test_images(dataset_key)
    return (None, None, "", None, None, [], np.zeros((CANVAS, CANVAS), dtype=np.float32), [], 0,
            "⬅️ Đã dọn dẹp ảnh/mask hiện tại. Chọn ảnh mới rồi nhấn 'Tải ảnh'.",
            gr.update(visible=False), gr.update(visible=False),
            gr.update(choices=names, value=names[0] if names else None))


with gr.Blocks(title="Demo Tương Tác PGA-UNet") as demo:
    gr.Markdown("""
    # 🩻 Demo Tương Tác — PGA-UNet (Prompt-Guided Attention U-Net)
    Chọn bộ dữ liệu và ảnh test có sẵn Ground Truth, click 2 điểm để khoanh Box tổn thương,
    xem mô hình phân đoạn và đối chiếu 6 độ đo với Ground Truth. Có thể thêm nhiều vòng câu nhắc
    (**Tiếp tục**) trước khi chuyển sang ảnh khác (**Kết thúc**).
    """)

    st_canvas_img = gr.State(None)
    st_canvas_gt  = gr.State(None)
    st_points     = gr.State([])
    st_acc_mask   = gr.State(np.zeros((CANVAS, CANVAS), dtype=np.float32))
    st_boxes      = gr.State([])
    st_round      = gr.State(0)

    with gr.Row():
        ds_dd     = gr.Dropdown(choices=list(DATASET_DIRS.keys()), value='BTXRD',
                                 label="1. Ảnh này thuộc bộ dữ liệu nào?")
        img_dd    = gr.Dropdown(choices=list_test_images('BTXRD'), label="2. Ảnh test (có Ground Truth)")
        weight_dd = gr.Dropdown(choices=list(DATASET_DIRS.keys()), value='BTXRD',
                                 label="3. Bộ trọng số PGA-UNet dùng suy luận")
        load_btn  = gr.Button("📥 Tải ảnh", variant="primary")

    status_tb = gr.Textbox(value=EMPTY_CANVAS_MSG, label="Trạng thái", interactive=False)

    with gr.Row():
        with gr.Column():
            canvas_display = gr.Image(label="3. Canvas 512×512 — click 2 điểm để khoanh Box", type="numpy")
            with gr.Row():
                confirm_btn   = gr.Button("✅ Xác Nhận Vùng Khoanh", variant="primary")
                reset_pts_btn = gr.Button("↺ Đặt lại điểm")
            show_gt_cb = gr.Checkbox(value=False, label="Hiện Ground Truth (viền xanh lá)")
        with gr.Column():
            result_display = gr.Image(label="4. Kết quả overlay (đỏ = dự đoán, xanh = GT)", type="numpy")
            metrics_md = gr.Markdown()
            with gr.Row():
                continue_btn = gr.Button("➕ Tiếp tục (thêm câu nhắc)", visible=False)
                end_btn      = gr.Button("🏁 Kết thúc (đổi ảnh khác)", visible=False, variant="stop")

    ds_dd.change(on_dataset_change, [ds_dd], [img_dd, weight_dd])

    load_btn.click(
        on_load_image, [ds_dd, img_dd],
        [canvas_display, st_canvas_img, st_canvas_gt, st_points, st_acc_mask, st_boxes, st_round,
         status_tb, result_display, metrics_md, continue_btn, end_btn]
    )

    canvas_display.select(
        on_click, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, st_points, show_gt_cb],
        [canvas_display, st_points, status_tb]
    )

    reset_pts_btn.click(
        on_reset_points, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, show_gt_cb],
        [canvas_display, st_points, status_tb]
    )

    show_gt_cb.change(
        on_toggle_gt, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, st_points, show_gt_cb],
        [canvas_display]
    )

    confirm_btn.click(
        on_confirm, [st_points, st_canvas_img, st_canvas_gt, st_acc_mask, st_boxes, weight_dd, st_round, show_gt_cb],
        [result_display, metrics_md, st_acc_mask, st_boxes, st_round,
         canvas_display, st_points, status_tb, continue_btn, end_btn]
    )

    continue_btn.click(
        on_continue, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, show_gt_cb],
        [canvas_display, st_points, status_tb, continue_btn, end_btn]
    )

    end_btn.click(
        on_end, [ds_dd],
        [canvas_display, result_display, metrics_md, st_canvas_img, st_canvas_gt, st_points,
         st_acc_mask, st_boxes, st_round, status_tb, continue_btn, end_btn, img_dd]
    )

demo.launch(debug=True, share=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ad86c74320e4b29ed7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
